In [5]:

from typing import Annotated

from langchain.agents import AgentState
from langchain_core.messages import ToolMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command


class CustomState(AgentState):
    username: str


@tool
def get_user_name(
        # 注入工具调用ID
        tool_call_id: Annotated[str, InjectedToolCallId],
        config: RunnableConfig
) -> Command:
    """获取当前用户名到状态"""
    user_name = config['configurable'].get('user_name', 'zs')
    # 用于更新AgentState
    return Command(update={
        "username": user_name,
        "messages": [
            # 表示工具调用成功 tool
            ToolMessage(
                content=f"成功的得到当前用户名{user_name}",
                tool_call_id=tool_call_id
            )
        ]
    })


@tool
def greet_user(state: Annotated[CustomState, InjectedState]):
    """鼓舞当前状态用户"""
    username = state['username']
    return f"祝贺你{username}，你已成功完成任务"

In [9]:
from langchain.agents import create_agent
import os
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='deepseek-ai/DeepSeek-V3.2',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

agent = create_agent(
    state_schema=CustomState,
    tools=[get_user_name, greet_user],
    model=model,
)


In [10]:
from langchain_core.messages import HumanMessage

agent.invoke(input={"messages": [HumanMessage("开始鼓舞")]}, config={"configurable": {"user_name": "ls"}})

{'messages': [HumanMessage(content='开始鼓舞', additional_kwargs={}, response_metadata={}, id='c141d205-a100-4d66-9c58-bd4d489d8019'),
  AIMessage(content='我来帮您鼓舞用户！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 329, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 329}, 'model_provider': 'openai', 'model_name': 'deepseek-ai/DeepSeek-V3.2', 'system_fingerprint': '', 'id': '019dab373de68b7354e53f13bad0e6c4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dab37-3cf0-78d3-816b-3780f52cdd90-0', tool_calls=[{'name': 'greet_user', 'args': {}, 'id': '019dab37485c71982434be09a0968cfc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 329, '